# 04 — Embeddings (image/texte) et index FAISS

```text
Embeddings image (+ texte) et construction des index FAISS. L'index ne contient QUE la KB.

  python 04_embeddings_index.py                       # tous les encodeurs de config.yaml
  python 04_embeddings_index.py --encoders clip-b32   # un seul

Sorties (par encodeur E) :
  embeddings/E/{kb,val,test}_img.npy   float32, L2-normalisés  + ids_{split}.npy (ordre des lignes)
  embeddings/E/kb_txt.npy              (encodeurs avec tour texte : clip, siglip) 1 vecteur / caption KB
  embeddings/sbert_kb_txt.npy          Sentence-BERT sur caption_clean (1 vecteur / caption KB)
  index/E_flatip.faiss  index/E_hnsw.faiss  index/E_txt_flatip.faiss
  index/index_report.json              dimensions, métrique, taille, temps, recall HNSW vs exact, latence
```

> Projet : étude comparative de techniques RAG pour le captioning d'images (MS COCO) — **groupe G1**. Exécuter les notebooks dans l'ordre : 00 → 01 → 01b → 02 → … → 09. Les helpers partagés sont dans `common.py`, `visual.py`, `rag_data.py`.

In [13]:
%pip install sentencepiece protobuf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import os
# --- paramètre : "config.yaml" = vraies données COCO ; "config_test.yaml" = mini-COCO synthétique (test sans réseau)
CONFIG = os.environ.get("RAG_CONFIG", "config.yaml")
os.environ["RAG_CONFIG"] = CONFIG
print("config utilisée :", CONFIG)

config utilisée : config.yaml


## Imports et constantes

In [15]:
import argparse, time, zlib
import numpy as np
import pandas as pd
from PIL import Image
from common import get_logger, load_config, mkdirs, paths, read_jsonl, set_seed, write_json
from visual import named_colors
log = get_logger("embed")

### `l2n()`

In [16]:
def l2n(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

### `DebugEncoder()`
Encodeur factice déterministe (histogramme couleur + gradient) : sert UNIQUEMENT à tester la plomberie.

In [17]:
class DebugEncoder:
    """Encodeur factice déterministe (histogramme couleur + gradient) : sert UNIQUEMENT à tester la plomberie."""
    has_text = True

    def __init__(self, dim):
        self.dim = dim
        self.rng = np.random.RandomState(0)
        self.proj = self.rng.randn(64 + 16, dim).astype(np.float32)

    def encode_images(self, paths_):
        out = []
        for pth in paths_:
            a = np.asarray(Image.open(pth).convert("RGB").resize((32, 32)), dtype=np.float32) / 255.
            h = np.concatenate([np.histogram(a[..., c], bins=8, range=(0, 1))[0] for c in range(3)] + [
                np.histogram(a.mean(2), bins=8, range=(0, 1))[0], np.histogram(np.abs(np.diff(a.mean(2), axis=0)), bins=8, range=(0, .5))[0],
                np.histogram(np.abs(np.diff(a.mean(2), axis=1)), bins=8, range=(0, .5))[0], np.zeros(32)]).astype(np.float32)[:80]
            out.append(h / 1024.0)
        return l2n(np.array(out) @ self.proj)

    def encode_texts(self, texts):
        out = np.zeros((len(texts), 80), dtype=np.float32)
        for i, t in enumerate(texts):
            for w in t.lower().split():
                out[i, zlib.crc32(w.encode()) % 80] += 1
        return l2n(out @ self.proj)

### `_as_features()`
Selon la version de `transformers`, get_*_features renvoie un tenseur (<5) ou un objet de sortie

In [18]:
def _as_features(out, proj=None, dim=None):
    """Selon la version de `transformers`, get_*_features renvoie un tenseur (<5) ou un objet de sortie
    (BaseModelOutputWithPooling, >=5). On récupère toujours un tenseur, et on applique la projection CLIP si besoin."""
    f = out
    if not hasattr(f, "float"):                       # ce n'est pas un tenseur
        if getattr(f, "pooler_output", None) is not None:
            f = f.pooler_output
        elif isinstance(f, (tuple, list)):
            f = f[0]
        else:
            f = f.last_hidden_state[:, 0]
    if proj is not None and dim is not None and f.shape[-1] != dim:
        f = proj(f)                                   # sortie pas encore projetée dans l'espace commun image/texte
    return f

### `HFEncoder()`

In [19]:
class HFEncoder:
    def __init__(self, kind, model, batch, device=None):
        import torch
        from transformers import AutoImageProcessor, AutoModel, AutoProcessor, AutoTokenizer
        self.torch, self.kind, self.batch = torch, kind, batch
        self.dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = AutoModel.from_pretrained(model).to(self.dev).eval()
        self.vproj = getattr(self.model, "visual_projection", None) if kind == "clip" else None
        self.tproj = getattr(self.model, "text_projection", None) if kind == "clip" else None
        self.pdim = getattr(self.model.config, "projection_dim", None) if kind == "clip" else None
        if kind in ("clip", "siglip"):
            self.proc = AutoProcessor.from_pretrained(model)
            self.tok = AutoTokenizer.from_pretrained(model)
            self.has_text = True
        else:
            self.proc = AutoImageProcessor.from_pretrained(model)
            self.has_text = False

    def encode_images(self, paths_):
        torch, out = self.torch, []
        for i in range(0, len(paths_), self.batch):
            ims = [Image.open(q).convert("RGB") for q in paths_[i:i + self.batch]]
            with torch.no_grad():
                if self.kind in ("clip", "siglip"):
                    px = self.proc(images=ims, return_tensors="pt")["pixel_values"].to(self.dev)
                    f = _as_features(self.model.get_image_features(pixel_values=px), self.vproj, self.pdim)
                else:                      # DINOv2 : token CLS
                    px = self.proc(images=ims, return_tensors="pt")["pixel_values"].to(self.dev)
                    f = self.model(pixel_values=px).last_hidden_state[:, 0]
            out.append(f.float().cpu().numpy())
        return l2n(np.concatenate(out))

    def encode_texts(self, texts):
        torch, out = self.torch, []
        for i in range(0, len(texts), 256):
            kw = dict(padding="max_length", truncation=True, max_length=64) if self.kind == "siglip" else dict(padding=True, truncation=True, max_length=77)
            t = self.tok(texts[i:i + 256], return_tensors="pt", **kw).to(self.dev)
            with torch.no_grad():
                f = _as_features(self.model.get_text_features(**t), self.tproj, self.pdim)
            out.append(f.float().cpu().numpy())
        return l2n(np.concatenate(out))

### `cached_encode()`
Cache PAR IDENTIFIANT : réutilise les lignes déjà calculées et ne calcule que les manquantes

In [20]:

def cached_encode(f_vec, f_ids, ids, encode_subset, force=False):
    """Reuse a complete cache without writing; atomically save missing rows, or recompute all with force=True."""
    import os
    import shutil
    import tempfile

    ids = np.asarray(ids)
    have = {}

    if not force and f_vec.exists() and f_ids.exists():
        old_ids = np.load(f_ids)
        old_vec = np.load(f_vec)

        if len(old_ids) == len(old_vec):
            have = {
                int(i): old_vec[k]
                for k, i in enumerate(old_ids)
            }

    miss = [
        k for k, i in enumerate(ids)
        if force or int(i) not in have
    ]

    if miss:
        new_vectors = encode_subset(miss)

        if len(new_vectors) != len(miss):
            raise ValueError("Nombre d'embeddings calculés incorrect")

        for k, v in zip(miss, new_vectors):
            have[int(ids[k])] = v

    X = np.stack([have[int(i)] for i in ids]).astype(np.float32)
    if not miss:
        return X, len(ids), 0

    targets = [(f_vec, X), (f_ids, ids)]
    temp_files = {}
    backup_files = {}
    replaced = []

    try:
        for path, array in targets:
            try:
                fd, temp_path = tempfile.mkstemp(dir=path.parent, prefix=f".{path.name}.", suffix=".tmp")
                temp_files[path] = temp_path
                with os.fdopen(fd, "wb") as stream:
                    np.save(stream, array)
            except PermissionError as exc:
                raise PermissionError(f"Cannot write cache file '{path}'; it is probably read-only.") from exc

        for path, _ in targets:
            if path.exists():
                try:
                    fd, backup_path = tempfile.mkstemp(dir=path.parent, prefix=f".{path.name}.", suffix=".bak")
                    os.close(fd)
                    shutil.copyfile(path, backup_path)
                    backup_files[path] = backup_path
                except PermissionError as exc:
                    raise PermissionError(f"Cannot preserve cache file '{path}' before writing; it is probably read-only.") from exc

        for path, _ in targets:
            try:
                os.replace(temp_files[path], path)
                replaced.append(path)
                del temp_files[path]
            except PermissionError as exc:
                rollback_errors = []
                for replaced_path in reversed(replaced):
                    try:
                        if replaced_path in backup_files:
                            os.replace(backup_files[replaced_path], replaced_path)
                            del backup_files[replaced_path]
                        elif replaced_path.exists():
                            replaced_path.unlink()
                    except OSError as rollback_exc:
                        rollback_errors.append(f"{replaced_path}: {rollback_exc}")
                detail = f"Cannot write cache file '{path}'; it is probably read-only."
                if rollback_errors:
                    detail += " Rollback failed: " + "; ".join(rollback_errors)
                raise PermissionError(detail) from exc
    finally:
        for temp_path in list(temp_files.values()) + list(backup_files.values()):
            if os.path.exists(temp_path):
                os.unlink(temp_path)

    return X, len(ids) - len(miss), len(miss)


### `build_encoder()`

In [21]:
def build_encoder(name, ecfg):
    if ecfg["type"] == "debug":
        return DebugEncoder(ecfg["dim"])
    return HFEncoder(ecfg["type"], ecfg["model"], ecfg.get("batch", 64))

### `main()`

In [22]:
def main():
    import faiss
    ap = argparse.ArgumentParser(); ap.add_argument("--encoders", nargs="*"); ap.add_argument("--skip-sbert", action="store_true")
    args = ap.parse_args()
    cfg = load_config(); p = paths(cfg); mkdirs(p); set_seed(cfg["seed"])
    imgs = pd.read_csv(p["tables"] / "images.csv"); imgs = imgs[imgs.status == "ok"]
    caps = read_jsonl(p["tables"] / "kb_captions.jsonl")
    report = {}
    names = args.encoders or list(cfg["encoders"])
    kb_ids = imgs[imgs.split == "kb"].sort_values("image_id").image_id.to_numpy()
    for name in names:
        ecfg = cfg["encoders"][name]
        try:
            enc = build_encoder(name, ecfg)
        except Exception as e:                       # un encodeur optionnel qui ne se charge pas ne doit pas bloquer la release
            if name == cfg["primary_encoder"]:
                raise
            log.warning("encodeur %s IGNORÉ (%s: %s)", name, type(e).__name__, " ".join(str(e).split())[:200])
            report[name] = {"skipped": " ".join(str(e).split())[:300]}
            continue
        d = p["emb"] / name; d.mkdir(parents=True, exist_ok=True)
        rep = {"type": ecfg["type"], "model": ecfg.get("model", "debug"), "normalization": "L2 (float32)", "device": getattr(enc, "dev", "cpu")}
        tt = time.perf_counter(); reused = []
        for split in ["kb", "val", "test"]:
            sub = imgs[imgs.split == split].sort_values("image_id")
            pths = [str(p["art"] / q) for q in sub.path_rgb]
            X, n_old, n_new = cached_encode(d / f"{split}_img.npy", d / f"ids_{split}.npy", sub.image_id.to_numpy(),
                                            lambda ks: enc.encode_images([pths[k] for k in ks]))
            reused.append(f"{split}:{n_old}")
            log.info("%s/%s : %d embeddings réutilisés, %d calculés", name, split, n_old, n_new)
            rep[f"{split}_shape"] = list(X.shape)
            assert np.isfinite(X).all(), f"NaN dans {name}/{split}"
        rep["embed_images_seconds"] = round(time.perf_counter() - tt, 2)
        rep["image_embeddings_reused_from_cache"] = reused
        rep["dim"] = int(X.shape[1])
        # ---- index exact (cosinus = produit scalaire sur vecteurs normalisés)
        Xkb = np.load(d / "kb_img.npy")
        t0 = time.perf_counter()
        flat = faiss.IndexFlatIP(Xkb.shape[1]); flat.add(Xkb)
        rep["flat_index_build_seconds"] = round(time.perf_counter() - t0, 4)
        faiss.write_index(flat, str(p["index"] / f"{name}_flatip.faiss"))
        # ---- HNSW (documenté + mesuré face à l'exact)
        h = cfg["index"]["hnsw"]
        t0 = time.perf_counter()
        hn = faiss.IndexHNSWFlat(Xkb.shape[1], h["M"], faiss.METRIC_INNER_PRODUCT)
        hn.hnsw.efConstruction = h["efConstruction"]; hn.add(Xkb); hn.hnsw.efSearch = h["efSearch"]
        rep["hnsw_index_build_seconds"] = round(time.perf_counter() - t0, 4)
        faiss.write_index(hn, str(p["index"] / f"{name}_hnsw.faiss"))
        Q = np.load(d / "test_img.npy")
        t0 = time.perf_counter(); _, Ie = flat.search(Q, 10); te = (time.perf_counter() - t0) / len(Q) * 1000
        t0 = time.perf_counter(); _, Ih = hn.search(Q, 10); th = (time.perf_counter() - t0) / len(Q) * 1000
        rep["hnsw_recall@10_vs_exact"] = float(np.mean([len(set(a) & set(b)) / 10 for a, b in zip(Ie, Ih)]))
        rep["latency_ms_per_query"] = {"flat": round(te, 4), "hnsw": round(th, 4)}
        rep["index"] = {"type": "IndexFlatIP", "metric": "inner product on L2-normalised = cosine", "n_vectors": int(flat.ntotal),
                        "size_mb": round((p["index"] / f"{name}_flatip.faiss").stat().st_size / 1e6, 3),
                        "hnsw": {**h, "size_mb": round((p["index"] / f"{name}_hnsw.faiss").stat().st_size / 1e6, 3)}}
        # ---- texte (captions KB) avec la tour texte de l'encodeur, si elle existe
        if getattr(enc, "has_text", False):
            t0 = time.perf_counter()
            cap_ids = np.array([c["caption_id"] for c in caps]); texts = [c["caption_raw"] for c in caps]
            T, n_old, n_new = cached_encode(d / "kb_txt.npy", d / "kb_txt_caption_ids.npy", cap_ids,
                                            lambda ks: enc.encode_texts([texts[k] for k in ks]))
            log.info("%s : texte %d réutilisés, %d calculés", name, n_old, n_new)
            np.save(d / "kb_txt_image_ids.npy", np.array([c["image_id"] for c in caps]))
            ti = faiss.IndexFlatIP(T.shape[1]); ti.add(T)
            faiss.write_index(ti, str(p["index"] / f"{name}_txt_flatip.faiss"))
            rep["text_embed_seconds"] = round(time.perf_counter() - t0, 2); rep["text_shape"] = list(T.shape)
        report[name] = rep
        log.info("%s : dim=%d | kb=%d | HNSW recall@10=%.3f | %s", name, rep["dim"], len(Xkb), rep["hnsw_recall@10_vs_exact"], rep["latency_ms_per_query"])
        del enc

    # ---- Sentence-BERT (retrieval texte-texte, BM25 hybride, etc.)
    if not args.skip_sbert:
        try:
            cap_ids = np.array([c["caption_id"] for c in caps]); texts = [c["caption_clean"] for c in caps]

            def _sbert(ks):
                from sentence_transformers import SentenceTransformer
                m = SentenceTransformer(cfg["sbert_model"])
                return m.encode([texts[k] for k in ks], batch_size=256, normalize_embeddings=True, show_progress_bar=False)
            S, n_old, n_new = cached_encode(p["emb"] / "sbert_kb_txt.npy", p["emb"] / "sbert_kb_txt_caption_ids.npy", cap_ids, _sbert)
            log.info("SBERT : %d réutilisés, %d calculés", n_old, n_new)
            report["sbert"] = {"model": cfg["sbert_model"], "dim": int(S.shape[1]), "shape": list(S.shape)}
            log.info("SBERT ok %s", S.shape)
        except Exception as e:  # noqa
            log.warning("SBERT ignoré (%s: %s)", type(e).__name__, str(e)[:120])
            report["sbert"] = {"skipped": str(e)[:200]}
    old = {}
    try:
        from common import read_json
        old = read_json(p["index"] / "index_report.json")
    except Exception:
        pass
    old = {k: v for k, v in old.items() if k in cfg["encoders"] or k == "sbert"}   # retire les encodeurs supprimés de la config
    old.update(report)
    write_json(p["index"] / "index_report.json", old)

## Lancer l'étape

In [ ]:

import sys

sys.argv = ["notebook", "--encoders", "siglip-b16", "--skip-sbert"]

try:
    main()
except SystemExit as e:
    print("terminé, code de sortie :", e.code)


## Résultats

In [ ]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
from common import load_config, paths
cfg = load_config(); P = paths(cfg)

### Rapport d'index (dimension, métrique, taille, temps, latence)

In [ ]:
rep = json.load(open(P["index"] / "index_report.json"))
rows = {}
for k, v in rep.items():
    if k == "sbert" or "skipped" in v: continue
    rows[k] = {"dim": v["dim"], "n_kb": v["index"]["n_vectors"], "type": v["index"]["type"], "taille Mo": v["index"]["size_mb"],
               "build flat (s)": v["flat_index_build_seconds"], "build HNSW (s)": v["hnsw_index_build_seconds"],
               "recall@10 HNSW/exact": v["hnsw_recall@10_vs_exact"], "latence flat (ms)": v["latency_ms_per_query"]["flat"],
               "embed images (s)": v["embed_images_seconds"]}
display(pd.DataFrame(rows).T); print("SBERT :", rep.get("sbert"))

,dim,n_kb,type,taille Mo,build flat (s),build HNSW (s),recall@10 HNSW/exact,latence flat (ms),embed images (s)
clip-b32,512,10000,IndexFlatIP,20.48,0.0081,4.006,1.0,1.2502,0.42
dinov2-s,384,10000,IndexFlatIP,15.36,0.0497,6.8016,0.997,1.2185,0.83
debug-hist,96,10000,IndexFlatIP,3.84,0.0037,2.4069,0.999,0.3324,0.51


SBERT : {'model': 'sentence-transformers/all-MiniLM-L6-v2', 'dim': 384, 'shape': [50000, 384]}


### Contrôle des fichiers

In [ ]:
import numpy as np
d = P["emb"] / cfg["primary_encoder"]
for s in ["kb", "val", "test"]:
    X = np.load(d / f"{s}_img.npy"); print(s, X.shape, "norme moy.", float(np.linalg.norm(X, axis=1).mean()).__round__(4), "NaN:", bool(np.isnan(X).any()))

kb (10000, 512) norme moy. 1.0 NaN: False
val (100, 512) norme moy. 1.0 NaN: False
test (100, 512) norme moy. 1.0 NaN: False


In [ ]:
import sys
print(sys.version)
print(sys.executable)
import sentencepiece, transformers, faiss, torch
print("sentencepiece", sentencepiece.__version__)
print("transformers", transformers.__version__)

3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
c:\Users\Cerine\AppData\Local\Python\pythoncore-3.14-64\python.exe
sentencepiece 0.2.2
transformers 5.21.0


In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"